# EPL Speed Benchmark

比较 `jampy` 和 `jaxpy` 在同一组 EPL `sigma_los` 计算上的速度。这个 notebook 会把 `jaxpy` 的首次 JIT 编译时间和稳态运行时间分开统计。

如果要切换 `cpu/gpu`，先修改 `requested_backend`，然后重启 kernel 再运行。

In [ ]:
# Config + setup
import os
import sys
import importlib
import pickle as pkl
from pathlib import Path
from time import perf_counter

requested_backend = "cpu"  # change to "gpu" on the GPU server, then restart kernel
backend_platforms = {
    "cpu": "cpu",
    "gpu": "cuda,cpu",
    "tpu": "tpu,cpu",
}
os.environ["JAX_PLATFORMS"] = backend_platforms[requested_backend]

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np

jaxpy_root = Path.cwd().resolve()
jampy_root = jaxpy_root.parents[2] / "Jampy" / "jampy-8.1.4"
sys.path.insert(0, str(jaxpy_root))
sys.path.insert(0, str(jampy_root))

import param_util
from SLCOSMO import tool
from MGE_jax import MGE
from jam_axi_proj_jax import jam_axi_proj
import jampy.axi.jam_axi_proj as jam_base_mod

jam_base_mod = importlib.reload(jam_base_mod)
from jampy.axi.jam_axi_proj import jam_axi_proj as jam_base

print("Active JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("jaxpy root:", jaxpy_root)
print("jampy root:", jampy_root)

theta_E = 1.397
gamma = 2.027
q_mass = 0.946
z_lens = 0.222
z_source = 0.609
fwhm_psf = 0.6
grid_size = 60
n_repeat = 20

cosmology = {
    "Omegam": 0.3,
    "Omegak": 0.0,
    "w0": -1.0,
    "wa": 0.0,
    "h0": 70.0,
}

lens_light = pkl.load(open(jaxpy_root / "lens_light_jackpot.pkl", "rb"))[0]
surf_lum = jnp.asarray(lens_light["amp"])
sigma_lum = jnp.asarray(lens_light["sigma"])
_, qobs_lum = param_util.ellipticity2phi_q(lens_light["e1"], lens_light["e2"])
qobs_lum = jnp.asarray(qobs_lum)
beta = jnp.zeros_like(surf_lum)

x_lin = np.linspace(-2.0, 2.0, grid_size)
y_lin = np.linspace(-2.0, 2.0, grid_size)
x_mesh, y_mesh = np.meshgrid(x_lin, y_lin, indexing="xy")
xbin = x_mesh.ravel()
ybin = y_mesh.ravel()
goodbins = np.ones_like(xbin, dtype=bool)
pixsize = float(x_lin[1] - x_lin[0])
sigmapsf = jnp.array([fwhm_psf / 2.355])
normpsf = jnp.array([1.0])

distance = float(np.asarray(tool.dldsdls(z_lens, z_source, cosmology, n=20)[0]).ravel()[0])
mge_epl = MGE(
    tool.EPL_msunmpc,
    "thetaE",
    n_gauss=10,
    n_terms=28,
    sigma_start_mult=1 / 100,
    sigma_end_mult=50,
)
surf_pot, sigma_pot = mge_epl.decompose(
    thetaE=theta_E,
    gamma=gamma,
    zl=z_lens,
    zs=z_source,
    cosmology=cosmology,
)
qobs_pot = jnp.full_like(surf_pot, q_mass)

def derive_grid_params(x, y, qobs_lum, sigma_psf, pixsize):
    qmed = np.median(np.asarray(qobs_lum))
    rell = np.hypot(np.asarray(x), np.asarray(y) / qmed)
    step = float(np.min(np.asarray(sigma_psf)) / 4)
    mx = float(3 * np.max(np.asarray(sigma_psf)) + pixsize / np.sqrt(2))
    rmax = float(np.max(rell) + mx)
    nx = int(np.ceil(rmax / step))
    ny = int(np.ceil(rmax * qmed / step))
    nk = int(np.ceil(mx / step))
    return nx, ny, nk, step

def to_base_kwargs(kwargs):
    scalar_keys = {"inc", "mbh", "distance", "pixsize", "step"}
    pass_keys = {"logistic", "moment", "align", "ml"}
    out = {}
    for key, value in kwargs.items():
        if value is None:
            out[key] = value
        elif key in scalar_keys:
            out[key] = float(value)
        elif key in pass_keys:
            out[key] = value
        elif key == "goodbins":
            out[key] = np.asarray(value, dtype=bool)
        else:
            out[key] = np.asarray(value)
    return out

nx, ny, nk, step = derive_grid_params(xbin, ybin, qobs_lum, sigmapsf, pixsize)

common_kwargs = dict(
    surf_lum=surf_lum,
    sigma_lum=sigma_lum,
    qobs_lum=qobs_lum,
    surf_pot=surf_pot,
    sigma_pot=sigma_pot,
    qobs_pot=qobs_pot,
    inc=90.0,
    mbh=0.0,
    distance=distance,
    xbin=xbin,
    ybin=ybin,
    sigmapsf=sigmapsf,
    normpsf=normpsf,
    beta=beta,
    pixsize=pixsize,
    logistic=False,
    moment="zz",
    goodbins=goodbins,
    align="cyl",
    ml=None,
    nrad=20,
    nang=10,
    nlos=1500,
    step=step,
)
base_kwargs = to_base_kwargs(common_kwargs)
jax_kwargs = dict(common_kwargs, nx=nx, ny=ny, nk=nk)

print("model setup ready")
print(f"theta_E={theta_E}, gamma={gamma}, q={q_mass}, zl={z_lens}, zs={z_source}, FWHM={fwhm_psf}, grid={grid_size}x{grid_size}")


In [ ]:
# Benchmark
jax.clear_caches()
jax_obj = jam_axi_proj()

t0 = perf_counter()
sigma_jax_first, *_ = jax_obj.get_kinematics(**jax_kwargs, quiet=True)
jax.block_until_ready(sigma_jax_first)
jax_first_call = perf_counter() - t0

sigma_jax_settle, *_ = jax_obj.get_kinematics(**jax_kwargs, quiet=True)
jax.block_until_ready(sigma_jax_settle)

t0 = perf_counter()
for _ in range(n_repeat):
    sigma_base = jam_base(**base_kwargs, plot=False, quiet=True).model
base_total = perf_counter() - t0

t0 = perf_counter()
for _ in range(n_repeat):
    sigma_jax, *_ = jax_obj.get_kinematics(**jax_kwargs, quiet=True)
    jax.block_until_ready(sigma_jax)
jax_total = perf_counter() - t0

base_per_call = base_total / n_repeat
jax_per_call = jax_total / n_repeat
speedup_total = base_total / jax_total

base_ref = np.asarray(sigma_base)
jax_ref = np.asarray(sigma_jax)
abs_diff = np.abs(base_ref - jax_ref)

results = {
    "backend": jax.default_backend(),
    "grid_size": grid_size,
    "n_repeat": n_repeat,
    "jax_first_call_s": float(jax_first_call),
    "jampy_total_s": float(base_total),
    "jaxpy_total_s": float(jax_total),
    "jampy_per_call_s": float(base_per_call),
    "jaxpy_per_call_s": float(jax_per_call),
    "speedup_total": float(speedup_total),
    "max_abs_model_diff": float(abs_diff.max()),
}

print("benchmark setup: JAX first-call compile is excluded; steady-state timing uses one total timer over 20 calls")
print(f"jax first call (compile + execute): {jax_first_call:.6f} s")
print(f"jampy total over {n_repeat} calls: {base_total:.6f} s")
print(f"jaxpy total over {n_repeat} calls: {jax_total:.6f} s")
print(f"jampy avg per call: {base_per_call:.6f} s")
print(f"jaxpy avg per call: {jax_per_call:.6f} s")
print(f"steady-state speedup: {speedup_total:.3f}x")
print(f"max |model difference|: {abs_diff.max():.6g}")
results
